# 01 · export v1 — slice splits out of the merged pkl → per-split canonical CSV

**Kernel: `fttl-v1` (env-v1, Python 3.5).** Targets come from the transformed pkl, NOT the
raw pickle — upstream of the pkl the data is already clean (target defined, `cc_fttl==1`
dropped). The pkl was built as `train.append(test).append(val1).append(val2)` and the four
lengths are known (173758 / 43471 / 35254 / 51189), so the splits are recovered by slicing
row counts in append order. **Fill in on the company laptop: only the 3 `Z:` paths.**

Two 3.5-era constraints:
- **`src/config.py` cannot be imported here** (needs 3.7+) — paths/columns MIRROR
  `config.VERSIONS["v1"]` and `config.SPLITS` by hand. If config changes, change here too.
- **This pandas cannot write parquet** — output is CSV; convert in the analysis env
  (`notebook/scratch/csv_to_parquet_v1.ipynb`).

| writes (× train/test/val1/val2) | canonical columns |
|---|---|
| `inputs/raw_v1.csv` *(single file)* | raw extract as-is, id renamed |
| `inputs/features_v1_{split}.csv` | `claim_id` + post-preprocessing matrix (target kept) |
| `inputs/targets_v1_{split}.csv` | `claim_id, date, observed` |
| `detection/v1_scores_{split}.csv` | `claim_id, model_v1_score` |

Resolve these downstream via `config.split_path(kind, "v1", split)`.


In [ ]:
# -*- Python 3.5 compatible: no f-strings -*-
import os
import sys

import joblib
import pandas as pd

assert sys.version_info[:2] == (3, 5), "run this on the fttl-v1 kernel (env-v1)"

# ---- SOURCES: fill in the real Z: paths. These live HERE, not in config — a declared config
# ---- path would point the analysis .venv at a pickle it cannot open. -------------------
RAW_DATASET      = r"Z:\P10_...\inputs.pkl"              # raw extract
PROCESSED_INPUTS = r"Z:\...\inputs_transformed.pkl"      # 4 splits appended, post-preprocessing
PREDICTIONS      = r"Z:\...\predictions.pkl"             # [claimnumber, predictions]

# ---- canonical names (mirror of config.VERSIONS['v1']['columns']) ----
ID, DATE, OBSERVED, SCORE = "claimnumber", "lossdate", "veh_total_loss", "predictions"

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    parent = os.path.dirname(ROOT)
    assert parent != ROOT, "repo root not found above cwd"
    ROOT = parent
OUT = os.path.join(ROOT, "src", "data", "real")
print("repo root:", ROOT)


In [ ]:
# joblib.load, NOT pd.read_pickle: the Z: files are joblib dumps despite the .pkl extension
# (pd.read_pickle dies with "stack_global requires str"). joblib.load also opens plain
# pickles, so it is safe for every source here.
raw  = joblib.load(RAW_DATASET)
proc = joblib.load(PROCESSED_INPUTS)
pred = joblib.load(PREDICTIONS)
print("raw        ", raw.shape)
print("processed  ", proc.shape)
print("predictions", pred.shape)
print("\nprocessed columns:", list(proc.columns))

# the transformed table should carry the id (39 model-ready cols include claimnumber + target)
assert ID in proc.columns, "no '" + ID + "' in the transformed table — inspect and adjust"
assert ID in pred.columns and SCORE in pred.columns


In [ ]:
# ---- split boundaries are KNOWN (read off train.py's run, 2026-08-11) ------------------
# inputs_transformed.pkl was appended in this order: train | test | val1 | val2.
# Upstream of the pkl the data is already clean — target defined, cc_fttl==1 rows dropped —
# then filtered to the 39 training columns and pipeline-transformed. So slicing by row
# count fully recovers the splits.

N = {"train": 173758, "test": 43471, "val1": 35254, "val2": 51189}

TARGET = "target"          # target column name inside the transformed pkl — fix if different
assert TARGET in proc.columns, "no '" + TARGET + "' column — set TARGET to train.py's name"
assert sum(N.values()) == len(proc), (
    "split lengths sum to {0} but pkl has {1} rows".format(sum(N.values()), len(proc)))

# NOT a hard assert any more (2026-09-18): duplicate claim ids showed up here for the first
# time on a re-export, characterised in the "check duplicate" cell below as 45 ids / 90 rows,
# ALL fully-identical row pairs (0 with a genuine value difference) -- a harmless upstream
# merge artifact, not a data problem. `dedupe_full()` in the next cell is the real gate now:
# it collapses only fully-identical duplicates and hard-stops on anything else, applied AFTER
# positional slicing (never before -- slicing depends on `len(proc) == sum(N.values())`, which
# a pre-slice drop would break).
if not proc[ID].is_unique:
    print("{0} duplicate claim id row(s) in the transformed pkl -- see 'check duplicate' cell "
          "for the full characterisation; handled per-split by dedupe_full() below.".format(
          int(proc[ID].duplicated().sum())))


In [ ]:
# check duplicate -- characterise BEFORE deciding whether to drop anything: exact-duplicate
# rows are a harmless merge artifact, conflicting values are a real data problem, and duplicates
# that straddle two DIFFERENT positional splits are train/test leakage -- none of these are the
# same fix, so this only reports, it does not touch `proc`.
dup_mask = proc[ID].duplicated(keep=False)
n_dup_rows = int(dup_mask.sum())
n_dup_ids = int(proc.loc[dup_mask, ID].nunique())
print("duplicate rows: {0} / unique ids involved: {1}".format(n_dup_rows, n_dup_ids))

dup = proc[dup_mask].copy()
n_conflict = int((dup.groupby(ID)[TARGET].nunique(dropna=False) > 1).sum())
print("ids with conflicting '{0}' across their duplicate rows: {1}".format(TARGET, n_conflict))

# which positional split (train | test | val1 | val2, in append order) each duplicate row lands
# in -- SPLITS[s] = proc.iloc[i:i+N[s]] only cares about ROW POSITION, so this is the same
# windowing that step uses, just applied here first.
bounds = {}
i = 0
for s in ("train", "test", "val1", "val2"):
    bounds[s] = (i, i + N[s])
    i += N[s]


def which_split(pos):
    for s, (lo, hi) in bounds.items():
        if lo <= pos < hi:
            return s
    return None


pos_of = dict((idx, pos) for pos, idx in enumerate(proc.index))
dup["_split"] = [which_split(pos_of[idx]) for idx in dup.index]

spread = dup.groupby(ID)["_split"].apply(lambda s: tuple(sorted(set(s))))
print("\nhow duplicate ids spread across splits (single split vs crossing splits):")
print(spread.value_counts())

crossing = spread[spread.apply(lambda t: len(t) > 1)]
print("\nids duplicated ACROSS different splits (possible train/test leakage): {0}".format(
    len(crossing)))
if len(crossing):
    print(crossing.head(20))

# ---- full-row check: TARGET matching is not enough -- proc has 39 columns, so two rows can
# ---- share the same id AND the same target while still differing in some OTHER feature column.
# ---- Only a group where EVERY row is a pairwise duplicate of every other row (ID excluded) is
# ---- safe to collapse with drop_duplicates(subset=[ID]) later.
full_dup = dup.groupby(ID).apply(lambda g: g.drop(columns=[ID]).duplicated(keep=False).all())
print("\nfull-column exact duplicate ids (safe to collapse): {0}".format(int(full_dup.sum())))
print("same id, some OTHER column differs (needs inspection): {0}".format(int((~full_dup).sum())))
if (~full_dup).any():
    print("example ids:", full_dup[~full_dup].index.tolist()[:10])


In [ ]:

def dedupe(df, value_col, label):
    # Duplicate claim ids inflate every downstream merge (score coverage > 1 was exactly
    # this). Identical duplicate rows are safe to collapse; CONFLICTING values are a data
    # problem — refuse to export until inspected.
    if df[ID].is_unique:
        return df
    dup = df[df.duplicated(ID, keep=False)]
    n_conflict = int((dup.groupby(ID)[value_col].nunique(dropna=False) > 1).sum())
    print("{0}: {1} duplicated rows over {2} claim ids ({3} with conflicting {4})".format(
        label, len(dup), dup[ID].nunique(), n_conflict, value_col))
    assert n_conflict == 0, label + ": conflicting duplicates — inspect before exporting"
    out = df.drop_duplicates(subset=[ID])
    print("  deduplicated {0}: {1} -> {2} rows".format(label, len(df), len(out)))
    return out


def dedupe_full(df, label):
    # proc/SPLITS[s] has 39 columns, unlike pred/dates above (id + one value column) -- two rows
    # can share an id AND match on any ONE column while still differing in some OTHER feature, so
    # dedupe()'s single-column conflict check is not enough here. Only a group where EVERY row is
    # a pairwise duplicate of every other row (id excluded) is safe to collapse; anything else is
    # a genuinely different row sharing a claim number and must not be silently dropped.
    if df[ID].is_unique:
        return df
    dup = df[df.duplicated(ID, keep=False)]
    full = dup.groupby(ID).apply(lambda g: g.drop(columns=[ID]).duplicated(keep=False).all())
    n_ids = dup[ID].nunique()
    n_full = int(full.sum())
    print("{0}: {1} duplicated rows over {2} claim ids ({3} fully identical, {4} with a "
          "genuine value difference)".format(label, len(dup), n_ids, n_full, n_ids - n_full))
    assert n_full == n_ids, (
        label + ": " + str(n_ids - n_full) + " claim id(s) differ on more than the id -- "
        "inspect before exporting (ids: " + str(full[~full].index.tolist()[:10]) + ")")
    out = df.drop_duplicates(subset=[ID])
    print("  deduplicated {0}: {1} -> {2} rows".format(label, len(df), len(out)))
    return out


pred  = dedupe(pred, SCORE, "predictions")
dates = dedupe(raw[[ID, DATE]], DATE, "raw dates")

SPLITS = {}
i = 0
for s in ("train", "test", "val1", "val2"):
    SPLITS[s] = dedupe_full(proc.iloc[i:i + N[s]], "proc-" + s)
    i += N[s]

print({s: len(SPLITS[s]) for s in ("train", "test", "val1", "val2")})

In [ ]:
# ---- per-split export (mirror of config.split_path: `_{split}` before the extension) ----
def out_path(sub, name):
    d = os.path.join(OUT, sub)
    if not os.path.isdir(d):
        os.makedirs(d)
    return os.path.join(d, name)

WRITTEN = []
def save(df, sub, name):
    p = out_path(sub, name)
    df.to_csv(p, index=False)
    WRITTEN.append((p, len(df), list(df.columns)))
    print("wrote {0:<36} rows {1:>9}".format(os.path.join(sub, name), len(df)))

# save(raw.rename(columns={ID: "claim_id"}), "inputs", "raw_v1.csv")

for s in ("train", "test", "val1", "val2"):
    df = SPLITS[s]

    save(df.rename(columns={ID: "claim_id"}), "inputs", "features_v1_" + s + ".csv")

    t = df[[ID, TARGET]].merge(dates, on=ID, how="left")
    t = t[[ID, DATE, TARGET]].rename(
        columns={ID: "claim_id", DATE: "date", TARGET: "observed"})
    save(t, "inputs", "targets_v1_" + s + ".csv")

    sc = df[[ID]].merge(pred[[ID, SCORE]], on=ID, how="inner").rename(
        columns={ID: "claim_id", SCORE: "model_v1_score"})
    save(sc, "detection", "v1_scores_" + s + ".csv")
    print("  {0}: score coverage {1:.6f}".format(s, float(len(sc)) / len(df)))
    assert len(sc) <= len(df), s + ": score rows exceed split rows — duplicate ids slipped through"


# additional conversion
- in analysis `.venv` — CSV → parquet: run **`notebook/scratch/csv_to_parquet_v1.ipynb`**
  (streaming pyarrow conversion + row-count check + parquet load check; safe for large files)


In [ ]:
# -*- confirm export: chunked re-read of every written CSV, rows + columns vs in-memory --
for path, n_exp, cols_exp in WRITTEN:
    name = os.path.relpath(path, OUT)
    cols = list(pd.read_csv(path, nrows=0).columns)
    n = sum(len(c) for c in pd.read_csv(path, chunksize=200000))
    print("{0:<36} {1:>9.1f} MB  rows {2:>9}".format(
        name, os.path.getsize(path) / 1024.0 ** 2, n))
    assert n == n_exp, name + ": rows {0} != in-memory {1}".format(n, n_exp)
    assert cols == cols_exp, name + ": column mismatch"
print("\nall {0} CSVs OK".format(len(WRITTEN)))
